# Generate-then-Refine con CV interna (K=5) y 120 por categoría (R6)

Completa el **protocolo B** del OE2 para la única técnica que falta: Generate-then-Refine. En el protocolo B el hiperparámetro `C` de la Regresión Logística
se elige con una **validación cruzada interna de K=5 particiones del pool** (en lugar de un dev de ~90 oraciones), y para que esa elección sea honesta los datos sintéticos
de cada partición se generan **solo con las oraciones de entrenamiento de esa partición** (nunca con las retenidas). Por eso Claude tiene que generar de nuevo dentro de cada partición.

**Configuración (la misma que el resto de los resultados del protocolo B):** las **7 intenciones** (con DES), el **corpus original** (`corpus/corpus_shiwilu_final.csv`, sin reetiquetar) y la **normalización de siempre**
(minúsculas, sin puntuación, sin tildes ni ñ; **sin signos de interrogación**). Es decir, no se pasa ninguna opción especial a `validacion_cruzada_cv_interna.py`.

**Qué hace este cuaderno**
1. **Genera con Claude** (paso 4a, la parte que gasta): por fold, 5 particiones internas + el pool completo, 6 lotes de 20 por categoría (= 120 por categoría). Cada archivo se guarda en Drive apenas se produce, así que si Colab se desconecta basta con volver a correr la celda: retoma sola.
2. **Evalúa** (paso 4b) con 120, 80 y 40 por categoría (los lotes son acumulativos: 80 y 40 usan los primeros lotes, sin gastar más). El **120 es el nivel principal** (~85% de las oraciones reales, comparable con Mixup y Retrotraducción); 80 y 40 muestran si más volumen ayuda. Entrena con los 3 modelos y elige `C` por CV interna.
3. **Une** con sin aumento, Mixup y Retrotraducción (protocolo B, ya calculados en el repositorio) y saca las mismas tablas y curvas ROC.
4. Entrega **un solo `.zip`**.

**Costo.** ~155 archivos × 7 llamadas ≈ **1085 llamadas** a `claude-sonnet-4-6`, unos **US$8-10** (~US$0.008 por llamada; el paso 4a imprime el número exacto de llamadas pendientes antes de empezar). Tarda unas 2 a 4 horas: no cierres la pestaña.

**Antes de empezar, haz `git push` de todo lo nuevo** (el cuaderno clona tu repositorio): `validacion_cruzada_cv_interna.py`, `unir_predicciones.py`, `rutas.py`, `generate_then_refine.py`,
`analisis_intrinseco/marcadores_fuentes.csv`, `evaluacion/resultados/cv_interna/` y `tecnicas_aumento/salidas/en_linea_marcadores_fuentes/`. El paso 1 comprueba que estén.

## 0. Montar Google Drive (aquí queda guardado lo generado, los resultados y los logs)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 1. Clonar tu repo

In [ ]:
import os
GITHUB_USUARIO = "LhiaHC"

%cd /content
!rm -rf /content/shiwilu-tesis
!git clone https://github.com/{GITHUB_USUARIO}/shiwilu-tesis.git
%cd /content/shiwilu-tesis
!git log --oneline -3

R = '/content/shiwilu-tesis/oe2_aumento_de_datos'
CV_DIR = f'{R}/evaluacion/resultados/cv_interna'
requeridos = [
    f'{R}/evaluacion/validacion_cruzada_cv_interna.py',
    f'{R}/evaluacion/unir_predicciones.py',
    f'{R}/analisis_intrinseco/marcadores_fuentes.csv',
    f'{R}/tecnicas_aumento/salidas/en_linea_marcadores_fuentes/generate_then_refine/fold0_pool_l1.csv',
] + [f'{CV_DIR}/validacion_cruzada_predicciones_sin_puntuacion_cv_interna_{t}.csv' for t in ('sin_aumento', 'mixup', 'retrotraduccion')]
faltan = [r for r in requeridos if not os.path.exists(r)]
assert not faltan, ('Este clon no tiene archivos nuevos: haz git add / commit / push de todo lo nuevo y vuelve a correr esta celda. Faltan: ' + ', '.join(faltan))
assert '--solo-generar' in open(requeridos[0], encoding='utf-8').read(), 'validacion_cruzada_cv_interna.py esta desactualizado en GitHub: haz push.'
assert 'INTENTOS_RESPUESTA' in open(f'{R}/tecnicas_aumento/generate_then_refine.py', encoding='utf-8').read(), 'generate_then_refine.py esta desactualizado en GitHub: haz push.'
print('OK')

## 2. Instalar dependencias

In [ ]:
!pip install -q anthropic python-dotenv sentence-transformers scikit-learn
import torch
print('GPU disponible:', torch.cuda.is_available())

## 3. API key (desde los Secretos de Colab) y prueba de conexión

La clave nunca se imprime. La prueba usa 10 tokens de salida.

In [ ]:
import os
from google.colab import userdata
import anthropic

os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")
cliente = anthropic.Anthropic(api_key=os.environ["ANTHROPIC_API_KEY"])
r = cliente.messages.create(model="claude-sonnet-4-6", max_tokens=10, messages=[{"role": "user", "content": "Responde solo: ok"}])
print("Conexion OK:", r.content[0].text)

## Configuración y respaldos (corre esta celda en cada sesión, antes del paso 4)

- `CANTIDAD`: oraciones por categoría que se generan (múltiplo de 20; 120 = 6 lotes).
- `NIVELES_EVAL`: con cuántas por categoría se evalúa (usan los primeros lotes de lo ya generado).
- `REGENERAR_LOTE0_POOL`: el lote 0 del pool (el de las corridas anteriores) se generó con el prompt anterior; con `True` se vuelve a generar con el prompt actual (~35 llamadas, ~US$0.3) para que todo el pool sea consistente.
- `PRUEBA = True`: prueba rápida y barata (solo el fold 0, 20 por categoría, caché aparte). Úsala primero si quieres comprobar que todo funciona (~US$0.3).

In [ ]:
import os, re, glob, shutil, subprocess, time

CANTIDAD = 120                  # oraciones por categoria que se generan
NIVELES_EVAL = [120, 80, 40]    # con cuantas por categoria se evalua (los lotes son acumulativos: 80 y 40 no gastan API; el 120 es el nivel principal)
REGENERAR_LOTE0_POOL = True     # volver a generar el lote 0 del pool con el prompt actual (~US$0.3)
PRUEBA = False                  # True = solo fold 0 y 20 por categoria, cache aparte (no contamina la corrida completa)
REEVALUAR = False               # True = ignora los respaldos de resultados y vuelve a evaluar los niveles

if PRUEBA:
    CANTIDAD, NIVELES_EVAL = 20, [20]
SUFIJO = "_prueba" if PRUEBA else ""
FOLDS = [0] if PRUEBA else [0, 1, 2, 3, 4]
N_LOTES = CANTIDAD // 20

CACHE = "/content/drive/MyDrive/shiwilu_cv_interna" + SUFIJO                      # lo generado (Claude)
RESPALDO_RES = "/content/drive/MyDrive/shiwilu_cv_interna_resultados" + SUFIJO     # CSV esenciales de cada nivel
DIR_LOGS = "/content/drive/MyDrive/shiwilu_cv_interna_logs"
GTR_CACHE = f"{CACHE}/generate_then_refine"
os.makedirs(GTR_CACHE, exist_ok=True); os.makedirs(RESPALDO_RES, exist_ok=True); os.makedirs(DIR_LOGS, exist_ok=True)
LOG = f"{DIR_LOGS}/gtr_cv_interna_{time.strftime('%Y%m%d_%H%M%S')}.txt"
SCRIPT_CV = "oe2_aumento_de_datos/evaluacion/validacion_cruzada_cv_interna.py"

ESENCIALES = ("validacion_cruzada_predicciones_sin_puntuacion_cv_interna_{tag}.csv", "validacion_cruzada_resumen_sin_puntuacion_cv_interna_{tag}.csv",
              "volumen_sintetico_sin_puntuacion_cv_interna_{tag}.csv", "cv_interna_f1_por_c_{tag}.csv")

def tag_de(nivel):
    return f"generate_then_refine_c{nivel}{SUFIJO}"

def correr(comando, cwd="/content/shiwilu-tesis"):
    # corre un comando mostrando la salida en vivo y guardandola (sin barras de progreso) en el log de Drive; se detiene si falla
    proc = subprocess.Popen(
        comando, cwd=cwd, text=True, encoding="utf-8", errors="replace", stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        env={**os.environ, "MPLBACKEND": "Agg", "TOKENIZERS_PARALLELISM": "false"},
    )
    with open(LOG, "a", encoding="utf-8") as log:
        for linea in proc.stdout:
            print(linea, end="")
            if "Batches:" not in linea and "Loading weights" not in linea:
                log.write(linea)
                log.flush()
    proc.wait()
    assert proc.returncode == 0, f"Fallo (codigo {proc.returncode}): {' '.join(comando)}"

def disponible(tag):
    # True si los CSV esenciales del nivel estan en esta sesion o, si no, en el respaldo de Drive (y entonces los restaura)
    locales = [f"{CV_DIR}/{p.format(tag=tag)}" for p in ESENCIALES]
    if all(os.path.exists(r) for r in locales):
        return True
    copias = [f"{RESPALDO_RES}/{p.format(tag=tag)}" for p in ESENCIALES]
    if all(os.path.exists(r) for r in copias):
        for r in copias:
            shutil.copy(r, CV_DIR)
        return True
    return False

def respaldar(tag):
    for p in ESENCIALES:
        shutil.copy(f"{CV_DIR}/{p.format(tag=tag)}", RESPALDO_RES)

# Siembra la cache con el pool que ya se genero (lotes 1-5 con el prompt actual; el lote 0 solo si no se va a regenerar), ya refiltrado con los marcadores de las fuentes
SEMILLA = f"{R}/tecnicas_aumento/salidas/en_linea_marcadores_fuentes/generate_then_refine"
for ruta_ in glob.glob(f"{SEMILLA}/fold*_pool*.csv"):
    nombre = os.path.basename(ruta_)
    es_lote0 = re.fullmatch(r"fold\d_pool\.csv", nombre) is not None
    if (es_lote0 and REGENERAR_LOTE0_POOL) or os.path.exists(f"{GTR_CACHE}/{nombre}"):
        continue
    shutil.copy(ruta_, GTR_CACHE)

print("CANTIDAD:", CANTIDAD, "| NIVELES_EVAL:", NIVELES_EVAL, "| PRUEBA:", PRUEBA, "| REGENERAR_LOTE0_POOL:", REGENERAR_LOTE0_POOL)
print("Log de esta sesion:", LOG)

## 4a. Generar con Claude (la parte que gasta)

Para cada fold: 5 particiones internas (cada una con solo sus oraciones de entrenamiento) + el pool completo, en lotes de 20 por categoría con otra ventana de ejemplos few-shot en cada lote.
Cada archivo `fold<N>_in<k>[_l<j>].csv` / `fold<N>_pool[_l<j>].csv` se guarda en Drive apenas se produce; lo que ya existe se reutiliza. Si Colab se desconecta, **vuelve a correr desde el paso de configuración y esta celda**: retoma donde quedó.

In [ ]:
def pendientes():
    faltan = []
    for f in FOLDS:
        for base in [f"fold{f}_in{k}" for k in range(5)] + [f"fold{f}_pool"]:
            for j in range(N_LOTES):
                nombre = f"{base}.csv" if j == 0 else f"{base}_l{j}.csv"
                if not os.path.exists(f"{GTR_CACHE}/{nombre}"):
                    faltan.append(nombre)
    return faltan

falta = pendientes()
print(f"Archivos por generar: {len(falta)} -> {len(falta) * 7} llamadas a la API, ~US${len(falta) * 7 * 0.008:.1f}")

for f in FOLDS:
    print(f"\n=============== Generando fold {f} ===============")
    if not any(n.startswith(f"fold{f}_") for n in falta):
        print(f"Fold {f}: ya esta todo generado; se salta.")
        continue
    try:
        correr(["python", SCRIPT_CV, "--tecnica", "generate_then_refine", "--cantidad", str(CANTIDAD),
                "--folds", str(f), "--cache", CACHE, "--solo-generar"])
    except AssertionError as error:
        print(f"\n*** El fold {f} fallo: {error}\n*** Lo ya generado sigue en Drive: vuelve a correr esta celda (retoma sola).")

falta = pendientes()
print("\nArchivos que aun faltan:", len(falta), falta[:5])

## 4b. Evaluar por nivel (sin gastar API: todo sale de la caché)

Para cada nivel de `NIVELES_EVAL`: une los lotes (quitando repetidas y copias), se queda con las aprobadas por los filtros, elige `C` por CV interna con los 3 modelos y predice el test.
Al terminar cada nivel respalda sus CSV esenciales en Drive; si ya estaba evaluado lo salta.

In [ ]:
assert not pendientes(), "Aun faltan archivos por generar: termina el paso 4a primero."
NIVELES_LISTOS = []
for nivel in NIVELES_EVAL:
    tag = tag_de(nivel)
    print(f"\n=============== Generate-then-Refine, {nivel} por categoria (CV interna K=5) ===============")
    if not REEVALUAR and disponible(tag):
        print(f"Nivel {nivel}: ya estaba evaluado (en esta sesion o en el respaldo de Drive); se reutiliza.")
        NIVELES_LISTOS.append(nivel)
        continue
    try:
        correr(["python", SCRIPT_CV, "--tecnica", "generate_then_refine", "--cantidad", str(nivel),
                "--cache", CACHE, "--etiqueta", f"c{nivel}{SUFIJO}"] + (["--folds", "0"] if PRUEBA else []))
        respaldar(tag)
        NIVELES_LISTOS.append(nivel)
    except AssertionError as error:
        print(f"\n*** El nivel {nivel} fallo y se omite: {error}\n*** Corrige y vuelve a correr esta celda (retoma sola).")
print("\nNiveles terminados:", NIVELES_LISTOS)

## 5. Unir con las otras tres técnicas, comparaciones pareadas y curvas ROC (por nivel)

In [ ]:
if not PRUEBA:
    import pandas as pd
    from IPython.display import display, Image

    NIVELES_LISTOS = [n for n in NIVELES_EVAL if disponible(tag_de(n))]   # tambien sirve tras una desconexion
    base = ["sin_aumento", "mixup", "retrotraduccion"]
    print("Niveles disponibles:", NIVELES_LISTOS)
    for nivel in NIVELES_LISTOS:
        etq = f"cv_interna_c{nivel}"
        entradas = [f"validacion_cruzada_predicciones_sin_puntuacion_cv_interna_{t}.csv" for t in base] + \
                   [f"validacion_cruzada_predicciones_sin_puntuacion_cv_interna_{tag_de(nivel)}.csv"]
        print("=" * 20, f"nivel {nivel}", "=" * 20)
        correr(["python", "oe2_aumento_de_datos/evaluacion/unir_predicciones.py", "--etiqueta", etq, "--entradas", *entradas])
        union = f"{CV_DIR}/validacion_cruzada_predicciones_sin_puntuacion_{etq}.csv"
        correr(["python", "oe2_aumento_de_datos/evaluacion/comparacion_pareada.py", "--entrada", union, "--etiqueta", etq, "--carpeta", CV_DIR])
        correr(["python", "oe2_aumento_de_datos/evaluacion/curvas_roc.py", "--entrada", union, "--etiqueta", etq, "--carpeta", CV_DIR])
        display(Image(f"{CV_DIR}/curvas_roc_{etq}.png"))

## 6. Los 12 experimentos con el protocolo B y la comparación frente a `sin_aumento`

In [ ]:
if not PRUEBA:
    columnas = {}
    ref = None
    for nivel in NIVELES_LISTOS:
        r_ = pd.read_csv(f"{CV_DIR}/validacion_cruzada_resumen_sin_puntuacion_cv_interna_c{nivel}.csv")
        f1 = lambda t: r_[r_.tecnica == t].set_index("modelo").f1_macro_agrupado
        if ref is None:
            ref = {"sin aumento": f1("sin_aumento"), "Mixup": f1("mixup"), "Retrotraduccion": f1("retrotraduccion")}
        ref[f"GtR {nivel}/categoria"] = f1("generate_then_refine")
    tabla = pd.DataFrame(ref).loc[["mbert", "labse", "xlmr"]]
    print("F1 macro agrupado, protocolo B (CV interna K=5):")
    display(tabla.round(4))

    filas = []
    for nivel in NIVELES_LISTOS:
        p_ = pd.read_csv(f"{CV_DIR}/comparacion_pareada_sin_puntuacion_cv_interna_c{nivel}.csv")
        for modelo in ["labse", "mbert", "xlmr"]:
            fila = p_[p_.comparacion == f"{modelo}: generate_then_refine - sin_aumento"].iloc[0]
            filas.append({"nivel": nivel, "modelo": modelo, "dif_vs_sin_aumento": round(fila.diferencia_f1, 4),
                          "IC95": f"[{fila.ic95_bajo:.3f}, {fila.ic95_alto:.3f}]", "distinguible": bool(fila.distinguible_de_cero)})
    print("Generate-then-Refine - sin aumento (diferencia pareada):")
    display(pd.DataFrame(filas))

    for nivel in NIVELES_LISTOS:
        V = pd.read_csv(f"{CV_DIR}/volumen_sintetico_sin_puntuacion_cv_interna_{tag_de(nivel)}.csv")
        C = pd.read_csv(f"{CV_DIR}/validacion_cruzada_predicciones_sin_puntuacion_cv_interna_{tag_de(nivel)}.csv")
        print(f"Nivel {nivel}: sinteticos con todo el pool = {V.pct_de_los_reales.mean():.1f}% de las reales | "
              f"C elegidos por modelo: { {m: sorted(g.C.unique().tolist()) for m, g in C.groupby('modelo')} }")

## 7. Descargar el resultado (un solo `.zip`)

Contiene `cv_interna/` (por nivel: predicciones, resumen, volumen, F1 de la CV por `C`; y el archivo unido con las 4 técnicas, comparaciones pareadas y curvas ROC) y `generate_then_refine/` (todo lo generado: particiones internas y pool).
Se descarga y además se copia a `Mi unidad/` por si la descarga falla. En tu computadora: `cv_interna/` va en `oe2_aumento_de_datos/evaluacion/resultados/cv_interna/` y `generate_then_refine/` en
`oe2_aumento_de_datos/tecnicas_aumento/salidas/en_linea_marcadores_fuentes/generate_then_refine/`.

In [ ]:
from google.colab import files

NIVELES_LISTOS = [n for n in NIVELES_EVAL if disponible(tag_de(n))]
carpeta = "/content/salida_gtr_cv_interna"
shutil.rmtree(carpeta, ignore_errors=True)
os.makedirs(carpeta + "/cv_interna")
shutil.copytree(GTR_CACHE, carpeta + "/generate_then_refine")        # lo generado
for ruta_ in glob.glob(f"{CV_DIR}/*generate_then_refine_c*") + glob.glob(f"{CV_DIR}/*cv_interna_c*"):   # resultados por nivel y archivos unidos
    destino = carpeta + "/cv_interna/" + os.path.basename(ruta_)
    shutil.copytree(ruta_, destino) if os.path.isdir(ruta_) else shutil.copy(ruta_, destino)
zip_path = shutil.make_archive("/content/salida_gtr_cv_interna", "zip", carpeta)
zip_drive = shutil.copy(zip_path, "/content/drive/MyDrive/salida_gtr_cv_interna" + SUFIJO + ".zip")   # copia de seguridad en Drive
print("Niveles incluidos:", NIVELES_LISTOS)
print("Copia del .zip en Drive:", zip_drive)
print(sorted(os.listdir(carpeta + "/cv_interna")))
try:
    files.download(zip_path)
except Exception as error:
    print("La descarga automatica fallo (", error, "). Descarga el .zip desde Drive:", zip_drive)